# Principle faithfulness — do Rechtssätze match the decisions that apply them?

An Austrian **Rechtssatz** is a court-distilled legal principle; RIS links it to the
decisions applying it (`from_rechtssatz`). This notebook asks a question only possible
because both genres are in one embedding space: **how close is the distilled principle to
the actual reasoning of its applying decisions — and is it *specifically* close to them,
or just close to family-law text in general?**

Method (all vectors already cached — zero new embedding):
- `faithfulness(RS)` = max cosine between the principle chunk and any chunk of its linked
  applying decisions (the best-matching passage);
- `baseline(RS)` = max cosine against an equal-sized random sample of *non*-applying
  decisions' chunks;
- the **specificity margin** = faithfulness − baseline.

**Topic-agnostic**: uses only the importer's RS↔decision links and the cached vectors —
runs unchanged for any topic corpus. Caveat: only decisions inside the keyword-matched
corpus count as "applying"; margins are corpus-relative.

## 1. Load cached vectors + RS↔decision links

In [ ]:
import json
import re
from pathlib import Path
import numpy as np

DATA_DIR, FIG_DIR, REPORT_DIR = Path("../data"), Path("../figures"), Path("../reports")
cache = np.load(DATA_DIR / "rag_echr_ris_emb_cache.npz", allow_pickle=True)
ids = [str(x) for x in cache["ids"]]
emb = cache["emb"].astype("float32")
id2row = {cid: i for i, cid in enumerate(ids)}
print(f"cached vectors: {emb.shape}")

ris = json.loads((DATA_DIR / "ris_parental_alienation.json").read_text())
rs_recs = {(r.get("rechtssatznummern") or "").strip(): r
           for r in ris if r.get("dokumenttyp") == "Rechtssatz"}
texts = [r for r in ris if r.get("dokumenttyp") == "Text"]

# chunk rows per document (RIS chunk ids are 'ris:<id>:<idx>')
doc_chunks = {}
for cid in ids:
    if cid.startswith("ris:"):
        doc_id = cid.split(":")[1]
        doc_chunks.setdefault(doc_id, []).append(id2row[cid])

# RS number -> applying decision doc-ids (via from_rechtssatz), restricted to indexed docs
applying = {}
for t in texts:
    for rsnum in (t.get("from_rechtssatz") or []):
        if t.get("id") in doc_chunks:
            applying.setdefault(rsnum, []).append(t.get("id"))

usable = {rs: docs for rs, docs in applying.items()
          if rs in rs_recs and rs_recs[rs].get("id") in doc_chunks}
print(f"Rechtssaetze with >=1 indexed applying decision: {len(usable)}")
print("applying decisions per RS:", {rs: len(d) for rs, d in sorted(usable.items())[:8]}, "...")

cached vectors: (42148, 768)
Rechtssaetze with >=1 indexed applying decision: 29
applying decisions per RS: {'RS0006893': 44, 'RS0007101': 124, 'RS0007272': 10, 'RS0007310': 24, 'RS0008614': 25, 'RS0047735': 9, 'RS0047934': 6, 'RS0047955': 23} ...


## 2. Faithfulness vs baseline

In [ ]:
rng = np.random.default_rng(42)
all_text_ids = [t.get("id") for t in texts if t.get("id") in doc_chunks]

rows = []
for rsnum, docs in usable.items():
    rs_vec = emb[doc_chunks[rs_recs[rsnum].get("id")][0]]        # principle = single chunk
    own_rows = [i for d in set(docs) for i in doc_chunks[d]]
    faith = float(np.max(emb[own_rows] @ rs_vec))
    # baseline: equal number of random NON-applying decisions
    others = [d for d in all_text_ids if d not in set(docs)]
    base_docs = rng.choice(others, size=min(len(set(docs)), len(others)), replace=False)
    base_rows = [i for d in base_docs for i in doc_chunks[d]]
    base = float(np.max(emb[base_rows] @ rs_vec))
    rows.append({"rs": rsnum, "n_applying": len(set(docs)),
                 "faithfulness": faith, "baseline": base, "margin": faith - base})

import pandas as pd
ft = pd.DataFrame(rows).sort_values("margin", ascending=False)
print(f"n = {len(ft)} principles\n")
print(f"faithfulness (max cos to own applying decisions): median {ft.faithfulness.median():.3f}")
print(f"baseline     (max cos to random other decisions): median {ft.baseline.median():.3f}")
print(f"specificity margin                              : median {ft.margin.median():.3f}")
print(f"principles closer to their own decisions than to random ones: "
      f"{(ft.margin > 0).sum()}/{len(ft)}\n")
print("most / least specific principles:")
print(ft.head(5)[["rs", "n_applying", "faithfulness", "baseline", "margin"]].round(3).to_string(index=False))
print("...")
print(ft.tail(3)[["rs", "n_applying", "faithfulness", "baseline", "margin"]].round(3).to_string(index=False))

n = 29 principles

faithfulness (max cos to own applying decisions): median 0.896
baseline     (max cos to random other decisions): median 0.863
specificity margin                              : median 0.028
principles closer to their own decisions than to random ones: 26/29

most / least specific principles:
       rs  n_applying  faithfulness  baseline  margin
RS0128436           2         0.958     0.866   0.092
RS0056290           4         0.853     0.798   0.055
RS0128809          25         0.940     0.886   0.053
RS0106454           8         0.905     0.854   0.051
RS0048343           4         0.893     0.849   0.044
...
       rs  n_applying  faithfulness  baseline  margin
RS0106313          28         0.848     0.848  -0.000
RS0047955          23         0.892     0.899  -0.007
RS0047934           6         0.846     0.863  -0.016


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(ft.baseline, ft.faithfulness, s=28 + 3 * ft.n_applying, color="#2c7fb8", alpha=0.8)
lims = [min(ft.baseline.min(), ft.faithfulness.min()) - 0.02,
        max(ft.baseline.max(), ft.faithfulness.max()) + 0.02]
ax.plot(lims, lims, "k--", lw=1, label="no specificity (margin = 0)")
for _, r in ft.head(3).iterrows():
    ax.annotate(r.rs, (r.baseline, r.faithfulness), fontsize=7,
                textcoords="offset points", xytext=(5, 3))
ax.set_xlabel("max cosine to RANDOM decisions (baseline)")
ax.set_ylabel("max cosine to OWN applying decisions (faithfulness)")
ax.set_title("Rechtssatz faithfulness: principles vs the decisions applying them\n"
             "(above the diagonal = specifically close to its own case-law; size = n applying)")
ax.legend(fontsize=8)
fig.tight_layout()
out = FIG_DIR / "ris_principle_faithfulness.png"
fig.savefig(out, dpi=130); plt.close(fig)
print("wrote", out.name)

wrote ris_principle_faithfulness.png


## 3. Report

In [ ]:
lines = ["# Principle faithfulness report (embedding-based, topic-agnostic)\n\n"]
lines.append(f"- n = **{len(ft)}** Rechtssaetze with indexed applying decisions.\n"
             f"- faithfulness (max cos, principle -> own applying decisions): median "
             f"**{ft.faithfulness.median():.3f}**\n"
             f"- baseline (max cos -> equal random sample of other decisions): median "
             f"**{ft.baseline.median():.3f}**\n"
             f"- specificity margin: median **{ft.margin.median():.3f}**; "
             f"**{(ft.margin > 0).sum()}/{len(ft)}** principles are closer to their own "
             f"case-law than to random family-law decisions.\n"
             f"- figure: `figures/ris_principle_faithfulness.png`\n\n"
             "## Reading\n"
             "- High faithfulness with a positive margin supports treating Rechtssaetze as "
             "**semantic surrogates** for their case-law in retrieval (why the `principle` "
             "genre scores highest precision) — measured, not assumed.\n"
             "- Near-zero margins flag principles whose corpus decisions discuss them only "
             "in passing — retrieval hits on such principles cite the RS, not the reasoning.\n"
             "- Topic-agnostic: RS-decision links + cached vectors only; corpus-relative "
             "(applying decisions outside the keyword corpus are invisible).\n")
out = REPORT_DIR / "principle_faithfulness_report.md"
out.write_text("".join(lines), encoding="utf-8")
print("wrote", out)

wrote ../reports/principle_faithfulness_report.md
